# Optimizer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# --- Konfiguration ---
TARGET_LINES = 30
X_MIN = 0.0
X_MAX = 0.01

# Vorbereitung der X-Achse für den Plot
x_vals = np.linspace(X_MIN, X_MAX, 100)
valid_lines_count = 0

plt.figure(figsize=(10, 6))

# --- Zufälliges Sampling & Filtern ---
while valid_lines_count < TARGET_LINES:
    # 1. Direktes Sampling innerhalb der physikalischen Grenzen
    betta_horst = random.uniform(0.3, 1.2)
    betta_abt = random.uniform(0.8, 1.5)
    
    # 2. Genaue Steigung (slope) und y-Achsenabschnitt (intercept) berechnen
    gap_horst = 0.00175
    gap_abt = 0.007
    
    slope = (betta_abt - betta_horst) / (gap_abt - gap_horst)
    intercept = betta_abt - (slope * gap_abt)
    
    # 3. Überprüfung der extremen Extrapolationsbedingung bei 0,3mm
    if (slope * 0.0003 + intercept) < 0.0:
        continue  # Verletzt die Extrapolationsregel, verwerfen und erneut versuchen
        
    # 4. Wenn bestanden, y-Werte berechnen und plotten
    y_vals = slope * x_vals + intercept
    plt.plot(x_vals, y_vals, color='seagreen', alpha=0.4)
    valid_lines_count += 1

# --- Visualisierungsebene (Plotten der Bedingungen & Grenzen) ---

# Sampling-Fenster 1: betta_horst bei 1,75mm (0,00175m)
plt.plot([0.00175, 0.00175], [0.3, 1.2], color='blue', linewidth=4, alpha=0.6, 
         label='Sampling-Fenster: betta_horst [0.3, 1.2]')

# Sampling-Fenster 2: betta_abt bei 7,0mm (0,007m)
plt.plot([0.007, 0.007], [0.8, 1.5], color='darkorange', linewidth=4, alpha=0.6, 
         label='Sampling-Fenster: betta_abt [0.8, 1.5]')

# Formatierung
plt.xlabel("space_between_frost [m]")
plt.ylabel("betta Korrektur Faktor [-]")
plt.xlim(X_MIN, X_MAX)
plt.xlim(0.0, 0.01)
plt.ylim(0.0, 2.0) 
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import sys
import os
import optuna

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from frost_evaporator import MultiExperimentAnalyzer, FrostEvaporatorSimulation, SimulationVisualizer

###########################################################################################
# Initialisation
###########################################################################################

cutoff_pct = 0.05

# --- Abt Setup (7.0 mm) ---
analyzer_abt = MultiExperimentAnalyzer(experiment_type='OptiAbt')
sim_abt = FrostEvaporatorSimulation("config_OptiAbt.yaml")
viz_abt = SimulationVisualizer(sim_abt.params)
path_exp_abt = Path(r"D:\mbc_nba\OptiAbt_Daten\Versuche\CSV_Export")
experiment_groups_abt = {
    "Abt-a": [56],
    "Abt-b": [40],
    "Abt-c": [28],
    # "Abt-d": [13],
}
cached_abt_data = {}
for name, ids in experiment_groups_abt.items():
    cached_abt_data[name] = analyzer_abt.analyze(exp_ids=ids, data_path=path_exp_abt, cutoff_pct=cutoff_pct, time_step=sim_abt.params.time_step)

abt_weights = {
    "Abt-a": 1.0,
    "Abt-b": 1.0, 
    "Abt-c": 1.0, 
    "Abt-d": 0.25
}

# --- Horst Setup (1.75 mm) ---
analyzer_horst = MultiExperimentAnalyzer(experiment_type='OptiHorst')
sim_horst      = FrostEvaporatorSimulation("config_OptiHorst.yaml")
viz_horst      = SimulationVisualizer(sim_horst.params)
path_exp_horst = Path(r"D:\mbc_nba\OptiHorst\Daten\Abtauen\CSV_Export")
experiment_groups_horst = {
    "Horst-a": [96],
    "Horst-b": [25],
    "Horst-c": [45],
    "Horst-d": [102],
    "Horst-e": [60],
}
cached_horst_data = {}
for name, ids in experiment_groups_horst.items():
    cached_horst_data[name] = analyzer_horst.analyze(exp_ids=ids, data_path=path_exp_horst, cutoff_pct=cutoff_pct, time_step=sim_horst.params.time_step)

horst_weights = {
    "Horst-a": 1.0, 
    "Horst-b": 1.0, 
    "Horst-c": 1.0, 
    "Horst-d": 1.0, 
    "Horst-e": 0.25,
}

###########################################################################################
# The Objective Function
###########################################################################################

def objective(trial):
    # 1. Suggest your variables
    betta_horst = trial.suggest_float('betta_1_75mm', 0.35, 0.5) 
    betta_abt = trial.suggest_float('betta_7_0mm', 0.9, 1.1)    
    
    gap_horst = 0.00175
    gap_abt = 0.007
    
    slope = (betta_abt - betta_horst) / (gap_abt - gap_horst)
    intercept = betta_abt - (slope * gap_abt)
    
    if (slope * 0.0003 + intercept) < 0.0:
        raise optuna.exceptions.TrialPruned()

    # Store suggested values in standard variables first
    h_conv_air_val      = trial.suggest_float('h_conv_air', 0.8, 1.1)
    surface_density_val = trial.suggest_float('surface_density', 0.7, 1.3)
    k_frost_val         = trial.suggest_float('k_frost', 0.7, 1.3)
    frost_diffusion_val = trial.suggest_float('frost_diffusion', 0.5, 2.0)

    # 2. Assign accepted factors for the simulation
    new_factors = {
        'h_conv_air':          h_conv_air_val, 
        'betta_intercept_air': intercept,
        'betta_slope_air':     slope,
        'surface_density':     surface_density_val, 
        'k_frost':             k_frost_val,
        'frost_diffusion':     frost_diffusion_val, 
        'pressure_loss':       0.734, 
    }

    print(new_factors)


    new_model_choices = {
        'frost_density_choice':      'hayashi_1977',
        'frost_conductivity_choice': 'yonko_sepsy_1967',
        'h_conv_air_choice':         'Wang',
    }

    # 3. Calculate the Deviation Penalty
    # List all the factors you want to keep close to 1.0
    penalized_factors = [
        betta_horst, betta_abt, h_conv_air_val, 
        surface_density_val, k_frost_val, frost_diffusion_val
    ]
    
    # Tuning this weight is important! 
    # If base error is ~10000, a weight of 5000 means a factor shifting from 1.0 to 1.5 
    # adds 4000 * (0.5)^2 = 1000 to the error.
    lambda_weight = 4000.0 
    
    deviation_penalty = lambda_weight * sum((val - 1.0)**2 for val in penalized_factors)

    total_summed_error = 0.0

    # =========================================================
    # Do OptiAbt Simulations (7.0 mm)
    # =========================================================
    try:
        sim_abt.update_correction_factors(new_factors, new_model_choices)
        
        for name, exp_data in cached_abt_data.items():
            states, inputs = sim_abt.run_validation(exp_data)
            exp_id = experiment_groups_abt[name][0] 

            use_melted_mass = (name == "Abt-a")
            
            df_err = viz_abt.get_relative_error_table(
                states, inputs, [exp_id], path_exp_abt, cutoff_pct, experiment_name="OptiAbt", use_melted_mass=use_melted_mass
            )

            row = df_err.loc[exp_id].fillna(0.0)
            print(row)

            # Only a and b have data for dp
            err_dp = row["dp"] if name in ["Abt-a", "Abt-b"] else 0.0
            
            # Sum of Squares of relative errors
            err_val = (err_dp**2) + (row["Q"]**2) + (row["m_frost"]**2)*0.5
            total_summed_error += err_val * abt_weights.get(name, 1.0) 
            
    except Exception as e:
        print(f"Trial {trial.number} failed in Abt: {e}")
        raise optuna.exceptions.TrialPruned()


    # =========================================================
    # Do OptiHorst Simulations (1.75 mm)
    # =========================================================
    try:
        sim_horst.update_correction_factors(new_factors, new_model_choices)
        
        for name, exp_data in cached_horst_data.items():
            states, inputs = sim_horst.run_validation(exp_data)
            exp_id = experiment_groups_horst[name][0]
            
            df_err = viz_horst.get_relative_error_table(
                states, inputs, [exp_id], path_exp_horst, cutoff_pct, experiment_name="OptiHorst"
            )
            
            row = df_err.loc[exp_id].fillna(0.0)
            print(row)
            
            # Sum of Squares of relative errors
            err_val = (row["dp"]**2) + (row["Q"]**2) + (row["m_frost"]**2)
            total_summed_error += err_val * horst_weights.get(name, 1.0)
            
    except Exception as e:
        print(f"Trial {trial.number} failed in Horst: {e}")
        raise optuna.exceptions.TrialPruned()


    # 4. Return the combined error
    error = total_summed_error + deviation_penalty
    error = min(error, 1e5)

    return error
###########################################################################################
# Optimization Execution
###########################################################################################

storage_name = "sqlite:///evaporator_optimization.db"

study = optuna.create_study(
    study_name="evaporator_tuning_full_v04", 
    storage=storage_name,
    load_if_exists=True,
    direction="minimize"
)

study.optimize(objective, n_trials=100)

print(f"Best factors: {study.best_params}")

# D:
# cd D:\mbc_nba\vclibpy\frost_evaporator_nba\notebooks
# optuna-dashboard sqlite:///evaporator_optimization.db

Can't copy file to new path: [Errno 13] Permission denied: 'd:\\mbc_nba\\vclibpy\\frost_evaporator_nba\\notebooks\\med_prop_Propane_REFPRP64.dll'
Can't copy file to new path: [Errno 13] Permission denied: 'd:\\mbc_nba\\vclibpy\\frost_evaporator_nba\\notebooks\\med_prop_R134a_REFPRP64.dll'


Model initialized with fluid: R134a
--- Aggregating Data (OptiAbt) for Experiments: [56] ---


d:\mbc_nba\vclibpy\frost_evaporator_nba\frost_evaporator\physics\fan_system_model.py:260: UserWarning: [Wang Correlation Validity] Transverse Pitch (Pt) value 50.00 is outside the valid range [17.7 - 31.75]. Results may be inaccurate.
  warnings.warn(


--- Aggregating Data (OptiAbt) for Experiments: [40] ---
--- Aggregating Data (OptiAbt) for Experiments: [28] ---
Model initialized with fluid: R32.FLD|R125.FLD
--- Aggregating Data (OptiHorst) for Experiments: [96] ---
--- Aggregating Data (OptiHorst) for Experiments: [25] ---
--- Aggregating Data (OptiHorst) for Experiments: [45] ---
--- Aggregating Data (OptiHorst) for Experiments: [102] ---
--- Aggregating Data (OptiHorst) for Experiments: [60] ---


[I 2026-03-05 13:42:37,554] A new study created in RDB with name: evaporator_tuning_full_v03
d:\mbc_nba\vclibpy\frost_evaporator_nba\frost_evaporator\physics\fan_system_model.py:260: UserWarning: [Wang Correlation Validity] Transverse Pitch (Pt) value 50.00 is outside the valid range [17.7 - 31.75]. Results may be inaccurate.
  warnings.warn(


{'h_conv_air': 1.0375497249954317, 'betta_intercept_air': 0.30202795374197244, 'betta_slope_air': 94.67764862185561, 'surface_density': 0.7566345680342376, 'k_frost': 1.1897079317885222, 'frost_diffusion': 1.520170032607549, 'pressure_loss': 0.734}
Model initialized with fluid: R134a
--- Simulation of Experiment Combined_1_OptiAbt ---
--- Priming Evaporator Model ---
--- Starting Simulation ---


Sim Combined_1_OptiAbt: 100%|██████████| 20/20 [00:20<00:00,  1.03s/it, Frost Surface Temp > 0°C in Layers 1, 2, 3, 4, 5]


dp           45.938485
h_ref_out    30.887516
Q            61.417313
m_frost      81.424218
Name: 56, dtype: float64
--- Simulation of Experiment Combined_1_OptiAbt ---
--- Priming Evaporator Model ---
--- Starting Simulation ---


Sim Combined_1_OptiAbt:  45%|████▌     | 9/20 [00:07<00:07,  1.46it/s, Frost Surface Temp > 0°C in Layers 1, 2]